# Création des embeddings avec OpenAI et sauvegarde dans un index FAISS

In [ ]:
# Importer les bibliothèques nécessaires
from langchain_openai import OpenAIEmbeddings # Pour générer des embeddings
import pandas as pd # Pour la manipulation des données
import numpy as np # Pour les opérations numériques
from tqdm import tqdm # Pour afficher une barre de progression
import os # Pour les opérations sur les fichiers
from dotenv import load_dotenv # Pour charger les variables d'environnement (clés API)

In [16]:
# 📁 Chemin vers les chunks créés par le notebook d'extraction
chunks_path = "../data/ready_for_embedding/chunks.csv"
df = pd.read_csv(chunks_path)

print(f"{len(df)} chunks chargés.")
df.head()

6346 chunks chargés.


,source,page,chunk_id,text
0,100 idees pour enseigner les ha - Mehdi Liratni,2,100 idees pour enseigner les ha - Mehdi Liratn...,Dr Mehdi Liratni Préface du Pr René Pry 100 ID...
1,100 idees pour enseigner les ha - Mehdi Liratni,3,100 idees pour enseigner les ha - Mehdi Liratn...,La version numérique de cet ouvrage a été réal...
2,100 idees pour enseigner les ha - Mehdi Liratni,4,100 idees pour enseigner les ha - Mehdi Liratn...,SOMMAIRE Préface Remerciements INTRODUCTION HA...
3,100 idees pour enseigner les ha - Mehdi Liratni,5,100 idees pour enseigner les ha - Mehdi Liratn...,CHAPITRE II COMPÉTENCES SOCLES DU COMPORTEMENT...
4,100 idees pour enseigner les ha - Mehdi Liratni,6,100 idees pour enseigner les ha - Mehdi Liratn...,Idée 26 Les 4 règles et objectifs importants d...


# Choix du modèle d'embedding

    ✅ OpenAI text-embedding-3-small

Ce modèle est optimal pour notre pipeline RAG car il est:
    - haute qualité pour le français
    - cohérent avec GPT-4o (même fournisseur)
    - 1536 dimensions (riche représentation)
    - **IDENTIQUE** à celui utilisé dans rag_module.py

In [ ]:
# Chargement du modèle d'embedding OpenAI (cohérent avec RAG)

from langchain_openai import OpenAIEmbeddings
import os
from dotenv import load_dotenv

# Charger la clé API
load_dotenv("../.env")
openai_api_key = os.getenv("OPENAI_API_KEY")

# Utiliser le même modèle d'embedding que dans rag_module.py
embeddings_model = OpenAIEmbeddings(
    api_key=openai_api_key,
    model="text-embedding-3-small"
)

print(f"Modèle OpenAI 'text-embedding-3-small' chargé.")
print(f"✅ Cohérent avec le pipeline RAG")

Modèle OpenAI 'text-embedding-3-small' chargé.
✅ Cohérent avec le pipeline RAG


In [ ]:
# Génération des embeddings avec OpenAI

texts = df["text"].tolist()

print(f"🔄 Génération de {len(texts)} embeddings avec OpenAI...")

# Générer les embeddings par batch pour éviter les limites d'API
embeddings_list = []
batch_size = 100  # Taille de batch raisonnable pour OpenAI

for i in tqdm(range(0, len(texts), batch_size), desc="Traitement par batch"):
    batch_texts = texts[i:i+batch_size]
    batch_embeddings = embeddings_model.embed_documents(batch_texts)
    embeddings_list.extend(batch_embeddings)

# Convertir en array numpy
embeddings = np.array(embeddings_list)

print(f"✅ {len(embeddings)} embeddings générés")
print(f"📊 Dimensions: {embeddings.shape}")

🔄 Génération de 6346 embeddings avec OpenAI...


Traitement par batch: 100%|██████████| 64/64 [01:03<00:00,  1.00it/s]


✅ 6346 embeddings générés
📊 Dimensions: (6346, 1536)


In [19]:
# --- Sauvegarde des embeddings et métadonnées ---

# 📁 Dossier de sortie
os.makedirs("../data/embeddings", exist_ok=True)

# Sauvegarde des vecteurs
np.save("../data/embeddings/chunks_embeddings.npy", embeddings)

# Sauvegarde des métadonnées
df.to_csv("../data/embeddings/chunks_metadata.csv", index=False)

print("✅ Embeddings et métadonnées sauvegardés.")


✅ Embeddings et métadonnées sauvegardés.


In [20]:
# Sanity check - Validation des embeddings OpenAI
print("📊 Dimensions des embeddings :", embeddings.shape)
print("🎯 Dimension attendue pour OpenAI text-embedding-3-small : 1536")
print("✅ Cohérence :", "OUI" if embeddings.shape[1] == 1536 else "❌ NON")
print("📝 Exemple de vecteur (10 premières valeurs) :", embeddings[0][:10])
print("📈 Plage de valeurs :", f"[{embeddings.min():.4f}, {embeddings.max():.4f}]")

📊 Dimensions des embeddings : (6346, 1536)
🎯 Dimension attendue pour OpenAI text-embedding-3-small : 1536
✅ Cohérence : OUI
📝 Exemple de vecteur (10 premières valeurs) : [ 0.01274838 -0.00144073 -0.00330525  0.01191655  0.00823635 -0.03239084
 -0.02020331  0.09150098 -0.02606392 -0.01604418]
📈 Plage de valeurs : [-0.1955, 0.1927]
